# TP final · Opción B · Comparación

Este notebook no entrena. Lee los runs que dejó `01_experimentos.ipynb` en `/Users/<usuario>/bigdata_emanuel_rb_tp_final` y elige el de mayor F1 macro de test.

Si un mismo modelo quedó más de una vez, se usa el run más reciente.


In [ ]:
import mlflow
import pandas as pd

usuario = spark.sql("SELECT current_user()").first()[0]
experimento = f"/Users/{usuario}/bigdata_emanuel_rb_tp_final"
exp = mlflow.get_experiment_by_name(experimento)
if exp is None:
    raise RuntimeError(f"No existe el experimento {experimento}. Corré primero 01_experimentos.ipynb.")

runs = mlflow.search_runs([exp.experiment_id])
if runs.empty:
    raise RuntimeError(f"El experimento {experimento} no tiene runs. Corré primero 01_experimentos.ipynb.")
runs = runs[runs["status"] == "FINISHED"].copy()
if "tags.mlflow.runName" in runs.columns:
    runs["run"] = runs["tags.mlflow.runName"]
else:
    runs["run"] = runs["run_name"]
runs = runs.sort_values("start_time", ascending=False)
esperados = ["logistica", "random_forest", "lightgbm", "catboost"]
runs = runs[runs["run"].isin(esperados)].drop_duplicates("run")
faltan = [nombre for nombre in esperados if nombre not in set(runs["run"])]
if faltan:
    raise RuntimeError(f"Faltan runs: {faltan}. Volvé a 01_experimentos.ipynb.")

columnas = {
    "run": "run",
    "metrics.test_f1_macro": "test_f1_macro",
    "metrics.train_f1_macro": "train_f1_macro",
    "metrics.test_accuracy": "test_accuracy",
    "metrics.test_f1_improve": "f1_improve",
    "metrics.test_f1_no_effect": "f1_no_effect",
    "metrics.test_f1_worsen": "f1_worsen",
    "params.puntajes_salud": "puntajes_salud",
    "params.train_filas": "train_filas",
    "params.test_filas": "test_filas",
}
comparacion = runs.rename(columns=columnas)[list(columnas.values())]
comparacion = comparacion.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
display(comparacion)

elegido = comparacion.iloc[0]
por_accuracy = comparacion.sort_values("test_accuracy", ascending=False).iloc[0]
print(
    f"Elegido: {elegido['run']} con F1 macro de test {float(elegido['test_f1_macro']):.4f}. "
    f"En train ese F1 es {float(elegido['train_f1_macro']):.4f}."
)
print(
    f"Por accuracy ganaría {por_accuracy['run']} "
    f"({float(por_accuracy['test_accuracy']):.4f}). "
    "La accuracy no decide: Improve es la clase mayoritaria."
)
print("experimento:", experimento)


## Por qué ese modelo

El criterio estaba fijado antes de mirar la tabla: gana el mayor F1 macro de test. Las tres clases pesan igual. `Improve` es cerca del 74 % de las respuestas, así que un modelo que casi siempre elige esa clase sube la accuracy y puede quedar abajo en F1 macro.

En la corrida local (scikit-learn 1.9.1, LightGBM 4.7.0, CatBoost 1.2.10, sin puntajes de salud) el orden por F1 macro fue `random_forest` 0.3891, `lightgbm` 0.3775, `catboost` 0.3713, `logistica` 0.3387. Por accuracy iba primero `lightgbm` (0.7260) y el bosque quedaba en 0.7192. El F1 de `Worsen` fue 0 en los cuatro: el test de ese corte tiene 3 filas de esa clase y ninguno las acertó. El número que vale es el de la tabla de arriba, leída de MLflow.


## Limitaciones

La encuesta salió de un Google Form en Reddit, Discord y redes. Quien respondió eligió entrar. El modelo no dice qué le pasa a alguien que no contestó este formulario.

`music_effects` y los puntajes de 0 a 10 son lo que la persona escribió, no un diagnóstico.

`Worsen` tiene 17 filas en toda la encuesta y alrededor de 3 en el test. Con tan pocas, el F1 de esa clase se mueve si cambia la semilla, y el orden entre modelos también.

Los árboles de esta configuración memorizan el train: el F1 macro de train llega a 1 y el de test queda mucho más abajo. Los hiperparámetros están fijos. La elección es el mejor F1 macro de test en este corte, no una búsqueda ni un resultado sobre la población.
